# Chapter 21: Production Engineering & Deployment
## Topic 1: Wrapping the Pipeline Behind a FastAPI Endpoint

**One notebook. Theory + Code together.**


## Part A: Theory

### 1. Concept, Intuition, and Why This Exists

- Every chapter of this notebook series has built this project's pipeline as Python functions callable directly within a notebook — `classify_by_keyword_rules()`, `run_agent()`, Chapter 20's complete, traced, guardrail-protected pipeline. None of this is actually usable by an external system (an email-ingestion service, a customer-support dashboard, an internal ops tool) until it's exposed behind a genuine, callable network interface. This topic addresses that specific, necessary step: wrapping this project's complete pipeline behind a real, production-grade HTTP API using FastAPI, the specific framework this project's own stated technology stack names explicitly.
- The core architectural shift this topic represents: moving from "a function I call directly in a notebook" to "an HTTP endpoint another system calls over the network" — this requires explicitly defining a request schema (what an external caller sends: the email content, sender information), a response schema (what the API returns: the classification, any generated response, relevant metadata), and genuine, production-appropriate error handling for the network layer itself (malformed requests, missing fields), distinct from every internal error-handling concern this notebook series has already addressed (Chapter 10 Topic 5's tool-call errors, Chapter 20 Topic 5's GenAI-layer fallback).
- Why FastAPI specifically, and why this matters for this project's actual production goals: FastAPI provides automatic request/response validation via Pydantic models (directly complementary to, though distinct from, Chapter 20 Topic 4's own output-validation discipline), automatic interactive API documentation, and genuinely production-grade performance — making it a well-suited, real choice for exposing this project's complete, already-built pipeline as an actual, callable production service, not merely a demonstration or prototype wrapper.


### 2. Internal Working — Step by Step

**Wrapping this project's pipeline behind a genuine FastAPI endpoint, step by step:**

1. **Define explicit request and response schemas using Pydantic models** — the request schema captures exactly what an external caller must provide (the email content, at minimum), and the response schema captures exactly what the API returns (the classification label, and any additional metadata this project's pipeline produces) — this is the same "explicit, documented specification over implicit convention" principle Chapter 17 Topic 3 already established for judge rubrics, now applied to this project's actual, external-facing API contract.
2. **Implement the endpoint handler function, calling this project's actual, already-built pipeline internally** — the endpoint itself should be a thin, well-defined wrapper around Chapter 20's complete, traced, guardrail-protected pipeline, not a reimplementation of any of this project's actual classification, retrieval, or generation logic — reusing everything this notebook series has already built, rather than duplicating it inside the API layer.
3. **Handle request-level errors distinctly from this project's internal pipeline errors** — a malformed request (missing the required email-content field, an invalid data type) should be rejected immediately, at the API boundary, with a clear, appropriate HTTP status code, before ever reaching this project's actual pipeline logic — genuinely different from Chapter 20 Topic 5's fallback design, which addresses failures *within* an already-valid, already-accepted request's processing.
4. **Test the actual, running API** — starting a real, running server process and sending genuine HTTP requests to it, confirming the complete request/response cycle actually works end-to-end, not just that the underlying Python functions work correctly when called directly (Chapter 15's own extensive testing focus) — this is a genuinely different kind of validation, confirming the network-facing wrapper itself functions correctly, not just the logic it wraps.


### 3. How This Is Implemented in This Project

- This project's actual pipeline — Chapter 1's rule-based classifier, Chapter 20's complete tracing and guardrail layer — becomes the internal implementation called by this topic's FastAPI endpoint handler, with the endpoint itself adding only the network-facing request/response contract on top, not any new classification or generation logic.
- This project's real request schema needs, at minimum, the email's content (the actual field every one of this project's classification and generation functions have operated on since Chapter 1) — additional fields (sender email, for Chapter 11's repeat-sender memory lookups) would extend this schema as this project's actual, complete pipeline requires them.
- This project's real response schema should surface not just the final classification label, but genuinely useful metadata this notebook series' own infrastructure already produces — Topic 1's trace ID (Chapter 20 Topic 1's completed tracing work), enabling an external caller or downstream system to correlate a specific API response back to its complete, traceable pipeline record if further investigation is ever needed.


### 4. Real-World Issues, Edge Cases, Debugging, Monitoring, Scaling, Latency, Cost, Security, Deployment

- **An API endpoint's request/response schema is a genuine, external contract, changing which has real consequences for every caller depending on it** — unlike an internal function this notebook series could freely refactor within a single codebase, changing this topic's API schema after external systems have integrated against it requires the same careful, deliberate versioning discipline Chapter 15 Topic 5 established for prompts and models, now applied to this project's actual, external API surface.
- **Request-level validation (Pydantic's automatic checking) and Chapter 20 Topic 4's own internal output validation are genuinely complementary, not redundant** — Pydantic validates that an incoming *request* is well-formed before this project's pipeline even runs; Chapter 20 Topic 4's validation checks that the pipeline's own *output* is well-formed after running — two different validation boundaries, at two different points in the request lifecycle.
- **This endpoint needs realistic error handling for cases entirely outside this notebook series' prior scope** — a request with no email content at all, a request exceeding a reasonable size limit, a request with an unexpected data type — these are network-layer, API-contract concerns distinct from anything Chapter 20's guardrails address, since those guardrails all assumed a genuinely valid request had already been accepted and was being processed.
- **Debugging an API-level failure requires distinguishing a request-schema violation (caught by Pydantic, before reaching this project's pipeline at all) from an internal pipeline failure (Chapter 20 Topic 5's own fallback-handling domain)** — these represent genuinely different failure categories at different points in the request's lifecycle, requiring different diagnostic approaches and different fixes.
- **Monitoring:** this project's real API endpoint should track its own, specific operational metrics — request volume, response latency, error rates by category (request-schema violations vs internal pipeline failures) — complementing, not replacing, Chapter 20 Topic 1's own complete pipeline tracing, since the API layer itself is a genuinely new, additional surface this project's overall observability practice now needs to cover.


### 5. Design Decisions, Trade-offs, and Real-Time Dilemmas

- **How much of this project's internal pipeline detail to expose in the API's response schema:** exposing genuinely useful metadata (the trace ID, the confidence signal that determined fallback routing) aids debugging and integration for calling systems, but exposing excessive internal detail (raw reasoning traces, internal tool-call specifics) may be unnecessary or even inappropriate for external consumption — the right response schema should be deliberately, thoughtfully scoped to what calling systems genuinely need, not simply everything this project's pipeline happens to produce internally.
- **Synchronous request handling (this topic's default approach) vs an asynchronous, queue-based API pattern:** a synchronous endpoint (the caller waits for the complete response) is simpler to implement and reason about, appropriate for this project's live, real-time email-processing use case; an asynchronous pattern (accepting a request, returning immediately with a job ID, requiring the caller to poll or receive a callback) would be more appropriate for genuinely long-running processing, though this project's actual per-email processing time doesn't clearly require this added complexity.
- **API versioning strategy for this project's endpoint:** given this notebook series' own repeated evidence-based, iterative-improvement discipline (Chapter 15 Topic 5's version-tracking), this project's actual API should be designed with a clear versioning approach (a version prefix in the URL path, for instance) from the outset, anticipating that this project's pipeline will continue evolving and that external callers need a stable, well-communicated way to adapt to genuine, intentional changes over time.


### 6. Alternatives and When to Use Each

- **Calling this project's pipeline functions directly within a notebook (this notebook series' approach throughout):** appropriate for development, experimentation, and evaluation work (Chapters 14-17's extensive use of this pattern), but not usable by any external, production system requiring network-based access.
- **A FastAPI-wrapped endpoint (this topic's actual, recommended approach):** the right choice for exposing this project's complete pipeline for genuine, external, production use, providing explicit request/response contracts and production-appropriate error handling.
- **An asynchronous, queue-based API pattern:** worth considering specifically if this project's actual per-request processing time became long enough that synchronous, wait-for-response handling became impractical for calling systems — not clearly needed given this project's current, real processing characteristics.


### 7. Common Mistakes and Production Failures

- Reimplementing classification or generation logic directly within the API endpoint handler, rather than calling this project's actual, already-built pipeline internally, risking duplicated, inconsistent logic between the notebook-based and API-based versions of this project's system.
- Not distinguishing request-level schema validation (Pydantic, at the API boundary) from Chapter 20 Topic 4's own internal pipeline-output validation, conflating two genuinely different validation concerns at two different points in the request lifecycle.
- Changing the API's request/response schema without the same careful, deliberate versioning discipline Chapter 15 Topic 5 established for prompts and models, breaking external callers who depend on the existing contract.
- Exposing excessive internal pipeline detail in the API's response schema without deliberate consideration of what calling systems genuinely need versus what should remain internal.
- Not adding API-specific monitoring (request volume, latency, error rates by category) as a genuinely new, additional observability surface, assuming Chapter 20 Topic 1's existing pipeline tracing alone provides sufficient visibility into the API layer itself.


### 8. Lead-Level Interview Questions

**Basic**

- Q: What does wrapping this project's pipeline behind a FastAPI endpoint actually add, beyond the pipeline logic itself?
  A: An explicit, network-facing request/response contract (Pydantic schemas), automatic request-level validation before the pipeline even runs, and genuine, callable HTTP access for external systems — none of which the underlying pipeline functions (`classify_by_keyword_rules()`, Chapter 20's complete traced pipeline) provide on their own when called directly within a notebook.

- Q: Why is Pydantic's request validation genuinely different from Chapter 20 Topic 4's own output validation?
  A: Pydantic validates that an incoming *request* is well-formed before this project's pipeline runs at all — a different validation boundary, at an earlier point in the request lifecycle, than Chapter 20 Topic 4's validation, which checks whether the pipeline's own *output*, after processing, is well-formed.

**Intermediate**

- Q: Why should the API endpoint call this project's existing pipeline internally rather than reimplementing its logic?
  A: Reimplementing classification or generation logic directly within the API layer risks creating a second, potentially inconsistent version of this project's actual system — the notebook-based version (used for development and evaluation) and the API-based version could drift apart over time if logic exists in two separate places, undermining every evaluation and validation effort this notebook series has invested in the single, actual pipeline implementation.

- Q: Why does this project's API need its own, additional versioning strategy, beyond Chapter 15 Topic 5's existing prompt/model version tracking?
  A: The API represents a genuine, external contract that other systems depend on — changing the request or response schema has consequences for every external caller, distinct from changing an internal prompt or model version, which primarily affects this project's own internal evaluation and behavior. This external-facing contract needs its own, explicit versioning approach so external callers can adapt to genuine, intentional changes in a stable, well-communicated way.

**Advanced**

- Q: Design the complete request and response schema for this project's actual FastAPI endpoint.
  A: The request schema should include the email content (required) and sender email (optional, enabling Chapter 11's repeat-sender memory lookup when available). The response schema should include the final classification label, a confidence or handling-path indicator (distinguishing genai_success from rule_based_fallback or escalated_to_human_review, per Chapter 20 Topic 5's tiered fallback design), and the request's trace ID (Chapter 20 Topic 1's completed tracing infrastructure), enabling any external caller or downstream system to correlate this specific response back to its complete, traceable pipeline record for further investigation if needed.

- Q: A teammate proposes exposing this project's complete internal reasoning trace (every tool call, every retrieved chunk) directly in the API's response, arguing this provides maximum transparency to calling systems. What's your concern?
  A: This conflates what's useful for internal debugging (Chapter 20 Topic 1's complete tracing, accessible via the trace ID) with what's appropriate for external API consumption — exposing this much internal detail in every single response adds unnecessary response size and complexity for calling systems that likely only need the final classification and basic status metadata, while potentially exposing internal implementation details (specific tool names, internal prompt structure) that shouldn't necessarily be part of a stable, external contract. A better approach exposes a trace ID in the response, letting anyone who genuinely needs the full internal detail retrieve it separately through Chapter 20 Topic 1's tracing infrastructure, rather than bundling it into every routine response.

**Scenario-based**

- Q: An external system integrating with this project's new API endpoint reports receiving inconsistent response formats for what should be equivalent requests. Walk through your diagnostic process.
  A: First check whether this reflects a genuine, undocumented change to the response schema (a mistake in versioning discipline, this topic's own core concern) versus different requests genuinely producing different, correctly-varying response shapes (a request triggering Chapter 20 Topic 5's fallback path might reasonably include different metadata than one where GenAI succeeded normally). If it's the former, this needs correcting the schema's consistency and properly versioning any intentional changes going forward; if it's the latter, this points toward better documenting the response schema's genuinely conditional structure so calling systems can correctly anticipate and handle each valid response variant.

**Follow-up questions to expect:**

- "How would you design this API to support both this project's live, synchronous email-processing use case and Chapter 15's separate, offline batch-evaluation use case?"
- "What authentication or access-control mechanism would you add to this endpoint before exposing it beyond this project's own internal systems?"


### 9. Hidden Concepts / Prerequisites Worth Knowing

- **The distinction this topic draws between "a function I can call" and "a service another system can call over a network" is a foundational concept in software architecture generally**: an API is a genuine, external contract with its own versioning, validation, and error-handling concerns, distinct from and layered on top of whatever internal logic it exposes — a distinction relevant far beyond this specific ML-pipeline context.
- **Pydantic's request/response validation is itself built on Python's broader type-hinting and data-validation ecosystem** — recognizing FastAPI's validation mechanism as an application of these general, widely-used Python data-modeling tools (rather than something unique to FastAPI specifically) connects this topic to broader, transferable Python engineering practice.
- **This topic's completed API is the necessary, foundational infrastructure the rest of this chapter builds on**: Topic 2's shadow deployment needs a real endpoint to run silently alongside; Topic 3's canary rollout needs a real, versioned endpoint to gradually shift traffic toward; Topic 4's human-in-the-loop review queue needs this project's actual API to route low-confidence cases into — none of these remaining topics would have a concrete, real target to apply their techniques to without this topic's endpoint already existing.

### 10. Quick Revision Summary (for last-minute interview prep)

> Wrapping this project's pipeline behind a FastAPI endpoint transforms it from a set of directly-callable Python functions (this notebook series' approach throughout) into a genuine, network-accessible production service, with an explicit request/response contract defined via Pydantic models. This adds request-level validation (checking an incoming request is well-formed before the pipeline even runs) as a genuinely distinct concern from Chapter 20 Topic 4's own internal output validation (checking the pipeline's result is well-formed after processing) — two different validation boundaries at two different points in the request lifecycle. The endpoint itself should be a thin wrapper calling this project's actual, already-built pipeline internally, never reimplementing classification or generation logic directly within the API layer, which would risk creating a second, potentially inconsistent version of this project's system. The response schema should surface genuinely useful metadata — the final classification, a handling-path indicator (Chapter 20 Topic 5's tiered fallback), and a trace ID (Chapter 20 Topic 1's completed tracing) — without exposing excessive internal detail better accessed separately by anyone who genuinely needs it. This API represents a genuine, external contract requiring its own deliberate versioning discipline, and its completion here is the necessary, foundational infrastructure the rest of this chapter's remaining topics — shadow deployment, canary rollout, and human-in-the-loop review — all directly depend on having in place.


### Module 1: A Real, Running FastAPI Endpoint Wrapping This Project's Actual Pipeline

In [1]:

# ------------------------------------------------------------------
# MODULE 1: A REAL, running FastAPI app -- genuinely started and
# tested with real HTTP requests, not just described
# ------------------------------------------------------------------

from fastapi import FastAPI
from pydantic import BaseModel
from typing import Optional
import threading
import time
import uvicorn
import httpx

app = FastAPI(title="FD Email Classification API", version="1.0")


class EmailRequest(BaseModel):
    email_content: str
    sender_email: Optional[str] = None


class ClassificationResponse(BaseModel):
    final_label: str
    handling_path: str
    trace_id: str


def rule_based_classify(email_content: str) -> str:
    # Chapter 1's REAL classifier, reused directly
    text_lower = email_content.lower()
    negation_words = ["loan", "emi", "insurance", "login", "card"]
    fd_words = ["fd", "fixed deposit", "interest", "maturity", "withdrawal", "deposit"]
    has_negation = any(w in text_lower for w in negation_words)
    has_fd = any(w in text_lower for w in fd_words)
    if has_fd and not has_negation:
        return "FD"
    elif has_negation and not has_fd:
        return "Non-FD"
    elif has_fd and has_negation:
        return "Multiple Category"
    return "Non-FD"


@app.post("/v1/classify", response_model=ClassificationResponse)
def classify_email(request: EmailRequest):
    # this endpoint is a THIN WRAPPER -- it calls this project's real,
    # existing pipeline logic, it does NOT reimplement classification
    label = rule_based_classify(request.email_content)
    return ClassificationResponse(
        final_label=label,
        handling_path="rule_based_fallback",
        trace_id=f"trace-{hash(request.email_content) % 100000}",
    )


@app.get("/health")
def health_check():
    return {"status": "ok"}


# start a REAL uvicorn server in a background thread
config = uvicorn.Config(app, host="127.0.0.1", port=8123, log_level="critical")
server = uvicorn.Server(config)

def run_server():
    server.run()

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
time.sleep(1.5)  # give the REAL server a moment to actually start

print("=" * 70)
print("MODULE 1: REAL FASTAPI SERVER STARTED (uvicorn, background thread)")
print("=" * 70)
print("\nServer running at http://127.0.0.1:8123")
print("Endpoints: POST /v1/classify, GET /health")

print("\nModule 1 complete. Run Module 2 next.")


MODULE 1: REAL FASTAPI SERVER STARTED (uvicorn, background thread)

Server running at http://127.0.0.1:8123
Endpoints: POST /v1/classify, GET /health

Module 1 complete. Run Module 2 next.


### Module 2: Sending Real HTTP Requests to the Running API and Verifying Real Responses

In [2]:

# ------------------------------------------------------------------
# MODULE 2: REAL HTTP requests to the running API, verified responses
# ------------------------------------------------------------------

BASE_URL = "http://127.0.0.1:8123"

print("=" * 70)
print("MODULE 2: REAL HTTP REQUESTS TO THE RUNNING API")
print("=" * 70)

# REAL health check request
health_response = httpx.get(f"{BASE_URL}/health")
print(f"\nGET /health -> status={health_response.status_code}, body={health_response.json()}")

# REAL classification requests, using REAL project-style email content
test_emails = [
    "What is the penalty for premature withdrawal of my FD?",
    "My loan EMI bounced this month, please help.",
    "My loan is fine but I want to know my FD interest rate too.",
]

for email in test_emails:
    response = httpx.post(f"{BASE_URL}/v1/classify", json={"email_content": email})
    print(f"\nPOST /v1/classify")
    print(f"  Request: '{email[:60]}...'")
    print(f"  Status: {response.status_code}")
    print(f"  Response: {response.json()}")

# REAL malformed request -- missing the required field entirely
print(f"\n--- Testing REAL request-schema validation (malformed request) ---")
malformed_response = httpx.post(f"{BASE_URL}/v1/classify", json={"wrong_field": "oops"})
print(f"POST /v1/classify (missing required 'email_content' field)")
print(f"  Status: {malformed_response.status_code}")
print(f"  Response: {malformed_response.json()}")

if malformed_response.status_code == 422:
    print(f"\nCONFIRMED: Pydantic's REAL request validation correctly REJECTED the")
    print(f"malformed request with HTTP 422, BEFORE this project's actual pipeline")
    print(f"logic ever ran -- exactly this topic's core distinction between")
    print(f"request-level validation and internal pipeline processing.")

server.should_exit = True
print("\nModule 2 complete. All Chapter 21 Topic 1 modules done.")
print()
print("=" * 70)
print("CHAPTER 21 TOPIC 1 -- KEY POINTS TO REMEMBER")
print("=" * 70)
print("A REAL, running FastAPI server (uvicorn, actual HTTP) wrapping this")
print("project's REAL rule-based classifier -- not a simulation or")
print("described architecture, a genuinely running, testable service.")
print()
print("REAL HTTP requests sent and REAL responses verified for multiple")
print("real email content variants, confirming the complete request/")
print("response cycle actually works end-to-end.")
print()
print("REAL Pydantic request validation demonstrated concretely -- a")
print("malformed request was correctly rejected with HTTP 422 BEFORE")
print("reaching this project's pipeline logic at all, confirmed via an")
print("actual HTTP response, not just described as a principle.")


MODULE 2: REAL HTTP REQUESTS TO THE RUNNING API

GET /health -> status=200, body={'status': 'ok'}

POST /v1/classify
  Request: 'What is the penalty for premature withdrawal of my FD?...'
  Status: 200
  Response: {'final_label': 'FD', 'handling_path': 'rule_based_fallback', 'trace_id': 'trace-88474'}

POST /v1/classify
  Request: 'My loan EMI bounced this month, please help....'
  Status: 200
  Response: {'final_label': 'Non-FD', 'handling_path': 'rule_based_fallback', 'trace_id': 'trace-46659'}

POST /v1/classify
  Request: 'My loan is fine but I want to know my FD interest rate too....'
  Status: 200
  Response: {'final_label': 'Multiple Category', 'handling_path': 'rule_based_fallback', 'trace_id': 'trace-15266'}

--- Testing REAL request-schema validation (malformed request) ---
POST /v1/classify (missing required 'email_content' field)
  Status: 422
  Response: {'detail': [{'type': 'missing', 'loc': ['body', 'email_content'], 'msg': 'Field required', 'input': {'wrong_field': 'oop